# LeetCode 2643 — Row With Maximum Ones bằng Python

Link đề: https://leetcode.com/problems/row-with-maximum-ones/ · Độ khó: **Easy** · Chủ đề: Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **tìm hàng có nhiều số `1` nhất** trong ma trận nhị phân và trả về **cặp `[chỉ số hàng, số lượng số 1]`**.
2. Hiểu quy tắc **hoà (tie-break)**: khi nhiều hàng cùng số `1` lớn nhất, chọn hàng có **chỉ số nhỏ nhất** — và biết cách để vòng lặp tự đảm bảo điều đó (so sánh `>` **chặt**, không dùng `>=`).
3. Dùng `sum(row)`, `row.count(1)`, `max(..., key=...)` và `enumerate`; biết `max` trả về phần tử **đầu tiên** khi có nhiều giá trị lớn nhất bằng nhau.
4. Viết 5 cách: đếm tay bằng hai vòng lặp, dùng `sum` / `count`, dùng `max` với `key`, so sánh tuple, và cách **dừng sớm** khi gặp hàng toàn số `1`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: dùng `>=` làm chọn hàng cuối thay vì đầu, khởi tạo `best_count = 0` mà bỏ sót ma trận toàn `0`, trả về sai thứ tự `[số 1, chỉ số]`, và nhầm hàng với cột.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `mat` kích thước `m × n` (chỉ chứa `0` và `1`). Hãy tìm **chỉ số hàng (từ 0)** có **nhiều số `1` nhất** và **số lượng số `1`** trong hàng đó.

Nếu có nhiều hàng cùng đạt số `1` lớn nhất, chọn hàng có **chỉ số nhỏ nhất**. Trả về mảng gồm hai phần tử: `[chỉ số hàng, số lượng số 1]`.

| Ví dụ | mat | Output | Giải thích |
|---|---|---|---|
| 1 | `[[0,1],[1,0]]` | `[0,1]` | Hai hàng đều có một số `1`; chọn hàng nhỏ hơn là `0`, số `1` là `1`. |
| 2 | `[[0,0,0],[0,1,1]]` | `[1,2]` | Hàng `1` có hai số `1`, nhiều nhất. |
| 3 | `[[0,0],[1,1],[0,0]]` | `[1,2]` | Hàng `1` có hai số `1`. |

Constraints:

- `m == mat.length`, `n == mat[i].length`
- `1 <= m, n <= 100`
- `mat[i][j]` là `0` hoặc `1`

Lưu ý: nếu **mọi** ô đều là `0` thì mọi hàng cùng có `0` số `1`; theo quy tắc hoà, đáp án là `[0, 0]`.

In [1]:
mat = [[0, 0, 0], [0, 1, 1]]
for i, row in enumerate(mat):
    print(f"hàng {i}: {row} -> có {sum(row)} số 1")

hàng 0: [0, 0, 0] -> có 0 số 1
hàng 1: [0, 1, 1] -> có 2 số 1


## 3. Kiến thức nền

### 3.1. Ma trận là danh sách các hàng

`mat[i]` là hàng `i` (một danh sách các `0/1`). `mat[i][j]` là ô ở hàng `i`, cột `j`. `len(mat)` là số hàng `m`; `len(mat[0])` là số cột `n`.

In [2]:
mat = [[0, 1], [1, 1], [0, 0]]
print(len(mat), len(mat[0]))      # 3 hàng, 2 cột
print(mat[1])                      # hàng 1: [1, 1]
print(mat[1][0])                   # ô (1, 0): 1

3 2
[1, 1]
1


### 3.2. Đếm số `1`: `sum` và `count`

Vì mỗi ô chỉ là `0` hoặc `1`, **tổng** các phần tử của hàng chính là số lượng số `1`. Hai cách tương đương:

- `sum(row)`: cộng các giá trị (`1` cộng thêm 1, `0` không đổi).
- `row.count(1)`: đếm số phần tử bằng `1`.

`sum` chỉ đúng khi dữ liệu thật sự là `0/1`; `count(1)` đúng cả khi hàng chứa giá trị khác.

In [3]:
row = [1, 0, 1, 1, 0]
print(sum(row))          # 3
print(row.count(1))      # 3
print(row.count(0))      # 2
print(sum([2, 0, 3]), [2, 0, 3].count(1))   # 5 0 : khác nhau khi dữ liệu không phải 0/1

3
3
2
5 0


### 3.3. `enumerate` và quy tắc so sánh chặt

`for i, row in enumerate(mat)` cho cả **chỉ số** `i` và **hàng** `row`. Khi tìm "lớn nhất đến giờ" ta so sánh số `1` của hàng mới với `best_count`:

- `if count > best_count:` — chỉ **thay** khi hàng mới **tốt hơn hẳn** ⇒ khi hoà, giữ hàng cũ (chỉ số nhỏ hơn). Đúng yêu cầu đề.
- `if count >= best_count:` — thay cả khi **bằng** ⇒ hàng sau (chỉ số lớn hơn) thắng. Sai yêu cầu đề.

In [4]:
counts = [2, 5, 5, 1]
best_i, best_c = 0, counts[0]
for i, c in enumerate(counts):
    if c > best_c:
        best_i, best_c = i, c
print("dùng >  :", best_i, best_c)     # 1 5  (giữ hàng đầu tiên đạt 5)

best_i, best_c = 0, counts[0]
for i, c in enumerate(counts):
    if c >= best_c:
        best_i, best_c = i, c
print("dùng >= :", best_i, best_c)     # 2 5  (bị hàng sau ghi đè)

dùng >  : 1 5
dùng >= : 2 5


### 3.4. `max` với `key` và quy tắc "đầu tiên thắng"

`max(iterable, key=f)` trả về phần tử có `f(x)` lớn nhất. Nếu nhiều phần tử cùng `f(x)` lớn nhất, **Python trả về phần tử xuất hiện đầu tiên**. Vậy `max(range(m), key=lambda i: sum(mat[i]))` cho **chỉ số hàng nhỏ nhất** trong các hàng đạt max — khớp quy tắc của đề.

In [5]:
counts = [2, 5, 5, 1]
print(max(range(len(counts)), key=lambda i: counts[i]))   # 1 (chứ không phải 2)
print(max([(2, "a"), (5, "b"), (5, "c")], key=lambda t: t[0]))   # (5, 'b')

1
(5, 'b')


### 3.5. So sánh tuple

Tuple được so sánh **từng phần tử từ trái sang phải**: `(5, -1) > (5, -2)` là `True` vì phần tử đầu bằng nhau, rồi `-1 > -2`. Mẹo: để "ưu tiên số `1` lớn, và khi hoà thì **chỉ số nhỏ**", ta so sánh `(số_1, -chỉ_số)` và lấy `max`.

In [6]:
print((5, -1) > (5, -2))      # True
print(max([(2, 0), (5, -1), (5, -2)]))   # (5, -1): cùng 5 số 1, chọn chỉ số 1 (vì -1 > -2)

True
(5, -1)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tưởng tượng một dãy các bạn xếp hàng, mỗi bạn cầm một dãy các lá cờ `0/1` (`1` là cờ đỏ). Ta cần tìm bạn cầm **nhiều cờ đỏ nhất**; nếu hoà thì chọn bạn **đứng trước**.

Cách làm tự nhiên:

1. Với mỗi hàng, **đếm** số `1`.
2. So với "kỷ lục hiện tại" — nếu **hơn hẳn** thì cập nhật kỷ lục và nhớ chỉ số hàng.
3. Cuối cùng trả `[chỉ số kỷ lục, số 1 của kỷ lục]`.

Tính tay ví dụ 2 `[[0,0,0],[0,1,1]]`: hàng 0 có `0` số `1`; hàng 1 có `2` số `1`. `2 > 0` ⇒ kỷ lục là hàng 1 với `2`. Đáp án `[1, 2]`.

In [7]:
mat = [[0, 0, 0], [0, 1, 1]]
record_idx, record = 0, -1
for i, row in enumerate(mat):
    c = sum(row)
    print(f"hàng {i}: {c} số 1; kỷ lục hiện tại = {record} (hàng {record_idx})")
    if c > record:
        record_idx, record = i, c
print([record_idx, record])

hàng 0: 0 số 1; kỷ lục hiện tại = -1 (hàng 0)
hàng 1: 2 số 1; kỷ lục hiện tại = 0 (hàng 0)
[1, 2]


## 5. Cách 1 — Brute Force / baseline: hai vòng lặp, đếm tay từng ô

**Ý tưởng:** dùng hai vòng lặp lồng nhau. Vòng ngoài qua từng hàng, vòng trong qua từng ô; mỗi lần thấy `1` thì tăng biến đếm của hàng. Hết hàng thì so sánh với kỷ lục (`>` chặt).

Khởi tạo `best_row = 0`, `best_count = 0` để xử lý luôn ma trận toàn `0` (đáp án `[0, 0]`).

In [8]:
from typing import List

class SolutionNested:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        best_row, best_count = 0, 0
        for i in range(len(mat)):
            count = 0
            for j in range(len(mat[i])):
                if mat[i][j] == 1:
                    count += 1
            if count > best_count:           # > chặt: hoà thì giữ hàng đầu
                best_row, best_count = i, count
        return [best_row, best_count]

In [9]:
sol = SolutionNested()
print(sol.rowAndMaximumOnes([[0, 1], [1, 0]]))               # [0, 1]
print(sol.rowAndMaximumOnes([[0, 0, 0], [0, 1, 1]]))         # [1, 2]
print(sol.rowAndMaximumOnes([[0, 0], [1, 1], [0, 0]]))       # [1, 2]
print(sol.rowAndMaximumOnes([[0, 0], [0, 0]]))               # [0, 0]

[0, 1]
[1, 2]
[1, 2]
[0, 0]


**Độ phức tạp:** mỗi ô được xem một lần ⇒ Time `O(m · n)`; Space `O(1)`.

**Nút thắt:** thuật toán đã tối ưu về bậc (phải đọc mọi ô để đếm đúng, trong trường hợp tổng quát). Tuy nhiên phần "đếm tay" dài dòng; ta thay bằng `sum` / `count`, và nhận ra có thể **dừng sớm** khi gặp hàng toàn `1`.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Đếm bằng hàm dựng sẵn.** Hàng nhị phân ⇒ `sum(row)` hoặc `row.count(1)` là số `1`. Bỏ vòng lặp trong cùng.
2. **Dùng `max` + `key`.** "Tìm phần tử lớn nhất theo một tiêu chí" là đúng việc của `max(..., key=...)`; tính chất "hoà thì lấy phần tử **đầu**" của Python khớp với yêu cầu của đề.
3. **Dừng sớm.** Một hàng có tối đa `n` số `1`. Nếu đã gặp hàng đạt `n`, không hàng nào phía sau có thể **hơn hẳn** (do tie-break chọn hàng đầu) ⇒ có thể `return` ngay.

In [10]:
mat = [[0, 1, 1], [1, 1, 1], [1, 0, 0]]
n = len(mat[0])
for i, row in enumerate(mat):
    print(f"hàng {i}: {sum(row)} số 1 / tối đa {n}", "-> toàn 1!" if sum(row) == n else "")

hàng 0: 2 số 1 / tối đa 3 
hàng 1: 3 số 1 / tối đa 3 -> toàn 1!
hàng 2: 1 số 1 / tối đa 3 


## 7. Các cách tiếp theo

### Cách 2 — `sum(row)` trong một vòng lặp

Giống Cách 1 nhưng vòng trong được thay bằng `sum(row)`. Các đoạn:

- `for i, row in enumerate(mat)`: lấy cả chỉ số và hàng.
- `count = sum(row)`: số `1` của hàng.
- `if count > best_count`: cập nhật khi hơn hẳn.

In [11]:
class SolutionSum:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        best_row, best_count = 0, 0
        for i, row in enumerate(mat):
            count = sum(row)
            if count > best_count:
                best_row, best_count = i, count
        return [best_row, best_count]

print(SolutionSum().rowAndMaximumOnes([[0, 1], [1, 0]]))          # [0, 1]
print(SolutionSum().rowAndMaximumOnes([[0, 0, 0], [0, 1, 1]]))    # [1, 2]

[0, 1]
[1, 2]


**Độ phức tạp:** Time `O(m · n)` (mỗi `sum` duyệt một hàng); Space `O(1)`.

### Cách 3 — `max(range(m), key=...)`

Tìm trực tiếp **chỉ số** hàng có `sum` lớn nhất; Python tự chọn chỉ số nhỏ nhất khi hoà. Sau đó trả `[idx, sum(mat[idx])]`.

In [12]:
class SolutionMaxKey:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        idx = max(range(len(mat)), key=lambda i: sum(mat[i]))
        return [idx, sum(mat[idx])]

print(SolutionMaxKey().rowAndMaximumOnes([[0, 1], [1, 0]]))          # [0, 1]
print(SolutionMaxKey().rowAndMaximumOnes([[0, 0], [1, 1], [0, 0]]))  # [1, 2]
print(SolutionMaxKey().rowAndMaximumOnes([[0, 0], [0, 0]]))          # [0, 0]

[0, 1]
[1, 2]
[0, 0]


**Độ phức tạp:** Time `O(m · n)` — chú ý `sum(mat[idx])` được tính lại một lần nữa (thêm `O(n)`), không đổi bậc; Space `O(1)`.

### Cách 4 — So sánh tuple `(số_1, -chỉ_số)`

Tạo từng tuple `(sum(row), -i)` rồi lấy `max`: tuple lớn nhất có số `1` lớn nhất và, khi hoà, `-i` lớn nhất tức `i` **nhỏ nhất**. Cách này **không phụ thuộc** vào quy tắc ngầm "max trả phần tử đầu" — tie-break được viết rõ ngay trong dữ liệu.

In [13]:
class SolutionTuple:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        ones, neg_idx = max((sum(row), -i) for i, row in enumerate(mat))
        return [-neg_idx, ones]

print(SolutionTuple().rowAndMaximumOnes([[0, 1], [1, 0]]))          # [0, 1]
print(SolutionTuple().rowAndMaximumOnes([[0, 0, 0], [0, 1, 1]]))    # [1, 2]

[0, 1]
[1, 2]


**Độ phức tạp:** Time `O(m · n)`; Space `O(1)` nếu dùng biểu thức sinh (không tạo danh sách trung gian).

### Cách 5 — Dừng sớm khi gặp hàng toàn `1`

Bổ sung vào Cách 2: nếu `count == n` thì không thể có hàng **hơn hẳn**, `return` ngay `[i, count]`. Lợi thế khi hàng toàn `1` xuất hiện sớm. Tệ nhất vẫn duyệt hết.

In [14]:
class SolutionEarlyStop:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        n = len(mat[0])
        best_row, best_count = 0, 0
        for i, row in enumerate(mat):
            count = sum(row)
            if count > best_count:
                best_row, best_count = i, count
                if count == n:               # không thể có hàng nào hơn hẳn
                    break
        return [best_row, best_count]

print(SolutionEarlyStop().rowAndMaximumOnes([[0, 1, 1], [1, 1, 1], [1, 1, 1]]))   # [1, 3]
print(SolutionEarlyStop().rowAndMaximumOnes([[0, 0], [0, 0]]))                    # [0, 0]

[1, 3]
[0, 0]


**Độ phức tạp:** Time `O(m · n)` tệ nhất, nhưng có thể dừng sau hàng đầu tiên toàn `1`; Space `O(1)`.

Lưu ý: nếu `n >= 1` và hàng toàn `1` chỉ được `break` khi nó **hơn hẳn** `best_count` (tức `count > best_count`) — đúng, vì nếu `best_count` đã là `n` thì ta đã `break` từ trước.

## 8. TRACE từng bước

Trace Cách 2 trên `mat = [[0,0,0],[0,1,1],[1,1,0],[0,0,1]]` (tie giữa hàng 1 và hàng 2):

| `i` | `row` | `count` | `count > best_count`? | `best_row` | `best_count` |
|---|---|---|---|---|---|
| (khởi tạo) | | | | 0 | 0 |
| 0 | `[0,0,0]` | 0 | `0 > 0` không | 0 | 0 |
| 1 | `[0,1,1]` | 2 | `2 > 0` có | 1 | 2 |
| 2 | `[1,1,0]` | 2 | `2 > 2` **không** (hoà, giữ hàng 1) | 1 | 2 |
| 3 | `[0,0,1]` | 1 | `1 > 2` không | 1 | 2 |

Kết quả `[1, 2]` — hàng 1 thắng hàng 2 nhờ tie-break.

In [15]:
def trace_row_max(mat):
    best_row, best_count = 0, 0
    print("khởi tạo: best_row=0, best_count=0")
    for i, row in enumerate(mat):
        count = sum(row)
        better = count > best_count
        print(f"i={i} row={row} count={count} count>best_count? {better}", end="")
        if better:
            best_row, best_count = i, count
            print(f" -> cập nhật: best_row={best_row}, best_count={best_count}")
        else:
            print(f" -> giữ: best_row={best_row}, best_count={best_count}")
    print("kết quả:", [best_row, best_count])
    return [best_row, best_count]

trace_row_max([[0, 0, 0], [0, 1, 1], [1, 1, 0], [0, 0, 1]])

khởi tạo: best_row=0, best_count=0
i=0 row=[0, 0, 0] count=0 count>best_count? False -> giữ: best_row=0, best_count=0
i=1 row=[0, 1, 1] count=2 count>best_count? True -> cập nhật: best_row=1, best_count=2
i=2 row=[1, 1, 0] count=2 count>best_count? False -> giữ: best_row=1, best_count=2
i=3 row=[0, 0, 1] count=1 count>best_count? False -> giữ: best_row=1, best_count=2
kết quả: [1, 2]


[1, 2]

In [16]:
trace_row_max([[0, 0], [0, 0]])      # ma trận toàn 0: không có hàng nào "hơn hẳn" 0

khởi tạo: best_row=0, best_count=0
i=0 row=[0, 0] count=0 count>best_count? False -> giữ: best_row=0, best_count=0
i=1 row=[0, 0] count=0 count>best_count? False -> giữ: best_row=0, best_count=0
kết quả: [0, 0]


[0, 0]

## 9. Code nộp LeetCode

Dùng Cách 2: `sum` + so sánh `>` chặt. Ngắn, rõ, tie-break đúng, xử lý được ma trận toàn `0`.

In [17]:
from typing import List

class Solution:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        best_row, best_count = 0, 0
        for i, row in enumerate(mat):
            count = sum(row)
            if count > best_count:
                best_row, best_count = i, count
        return [best_row, best_count]

In [18]:
print(Solution().rowAndMaximumOnes([[0, 1], [1, 0]]))            # [0, 1]
print(Solution().rowAndMaximumOnes([[0, 0, 0], [0, 1, 1]]))      # [1, 2]
print(Solution().rowAndMaximumOnes([[0, 0], [1, 1], [0, 0]]))    # [1, 2]

[0, 1]
[1, 2]
[1, 2]


## 10. Vì sao thuật toán đúng

1. **Invariant.** Sau khi xử lý các hàng `0..i`, `(best_row, best_count)` là hàng có số `1` lớn nhất trong các hàng đó; nếu có nhiều hàng đạt max thì `best_row` là chỉ số **nhỏ nhất** trong số đó. Ban đầu (chưa xử lý hàng nào) ta dùng `(0, 0)` là quy ước hợp lệ vì số `1` luôn `>= 0`.
2. **Bước quy nạp.** Hàng mới `i` có `count`. Nếu `count > best_count` thì hàng `i` là duy nhất đạt max mới ⇒ cập nhật. Nếu `count == best_count` thì hàng cũ có chỉ số nhỏ hơn (vì ta duyệt tăng dần) ⇒ **giữ nguyên** đúng tie-break. Nếu `count < best_count` thì không đổi.
3. **Ma trận toàn `0`.** Không bao giờ có `count > 0` ⇒ giữ `(0, 0)`; đề yêu cầu chọn hàng nhỏ nhất khi hoà nên `[0, 0]` là đáp án đúng.
4. **Số `1` của hàng.** Vì `mat[i][j] ∈ {0, 1}` nên `sum(row)` đúng bằng số phần tử bằng `1`.

## 11. Phân tích Time / Space Complexity

`m` = số hàng, `n` = số cột.

| Cách | Time | Space |
|---|---|---|
| 1. Hai vòng lặp, đếm tay | `O(m · n)` | `O(1)` |
| 2. `sum(row)` | `O(m · n)` | `O(1)` |
| 3. `max` với `key` | `O(m · n)` | `O(1)` |
| 4. So sánh tuple | `O(m · n)` | `O(1)` |
| 5. Dừng sớm | `O(m · n)` tệ nhất, thường ít hơn | `O(1)` |

Một ma trận bất kỳ buộc ta xem mọi ô (một ô `1` ẩn ở cuối có thể thay đổi đáp án), nên `O(m · n)` là tối ưu trong trường hợp tổng quát. Với `m, n <= 100`, tối đa `10^4` ô.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: hoà ở đầu / giữa / cuối; ma trận toàn `0`; toàn `1`; một hàng; một cột; `1×1`; số `1` lớn nhất ở hàng cuối.

In [19]:
sols = [SolutionNested(), SolutionSum(), SolutionMaxKey(), SolutionTuple(), SolutionEarlyStop(), Solution()]
cases = [
    ("ví dụ 1", [[0, 1], [1, 0]], [0, 1]),
    ("ví dụ 2", [[0, 0, 0], [0, 1, 1]], [1, 2]),
    ("ví dụ 3", [[0, 0], [1, 1], [0, 0]], [1, 2]),
    ("toàn 0", [[0, 0], [0, 0]], [0, 0]),
    ("toàn 1", [[1, 1], [1, 1]], [0, 2]),
    ("1x1 là 0", [[0]], [0, 0]),
    ("1x1 là 1", [[1]], [0, 1]),
    ("một hàng", [[1, 0, 1]], [0, 2]),
    ("một cột", [[0], [1], [1]], [1, 1]),
    ("hoà ở giữa và cuối", [[0, 0], [1, 0], [0, 1], [1, 0]], [1, 1]),
    ("max ở hàng cuối", [[0, 0, 1], [0, 1, 0], [1, 1, 1]], [2, 3]),
    ("hàng toàn 0 rồi toàn 1", [[0, 0, 0], [1, 1, 1], [1, 1, 1]], [1, 3]),
]
for name, mat, exp in cases:
    for s in sols:
        got = s.rowAndMaximumOnes([row[:] for row in mat])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> [0, 1]
PASS ví dụ 2 -> [1, 2]
PASS ví dụ 3 -> [1, 2]
PASS toàn 0 -> [0, 0]
PASS toàn 1 -> [0, 2]
PASS 1x1 là 0 -> [0, 0]
PASS 1x1 là 1 -> [0, 1]
PASS một hàng -> [0, 2]
PASS một cột -> [1, 1]
PASS hoà ở giữa và cuối -> [1, 1]
PASS max ở hàng cuối -> [2, 3]
PASS hàng toàn 0 rồi toàn 1 -> [1, 3]


### 12.2. Phản ví dụ cho lời giải sai dùng `>=`

Một cách sai hay gặp là dùng `>=` (xem mục 13). Ta tạo ca có hoà để lời giải sai thất bại, chứng minh bộ test **có khả năng bắt lỗi**.

In [20]:
def sai_dung_ge(mat):
    best_row, best_count = 0, 0
    for i, row in enumerate(mat):
        count = sum(row)
        if count >= best_count:      # SAI: hoà thì lấy hàng sau
            best_row, best_count = i, count
    return [best_row, best_count]

print(sai_dung_ge([[0, 1], [1, 0]]), "!=", Solution().rowAndMaximumOnes([[0, 1], [1, 0]]))
assert sai_dung_ge([[0, 1], [1, 0]]) != [0, 1]       # sai: trả [1, 1]

[1, 1] != [0, 1]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `mat` ngẫu nhiên đúng constraints (`1 <= m, n <= 100`, giá trị `0/1`) với xác suất `1` khác nhau (để hay xảy ra hoà). Oracle dùng định nghĩa: lấy các hàng đạt max rồi chọn `min(chỉ số)`.

In [21]:
import random
random.seed(2643)

def oracle(mat):
    counts = [row.count(1) for row in mat]
    mx = max(counts)
    idx = min(i for i, c in enumerate(counts) if c == mx)
    return [idx, mx]

for _ in range(500):
    m, n = random.randint(1, 10), random.randint(1, 10)
    p = random.choice([0.0, 0.1, 0.5, 0.9, 1.0])
    mat = [[1 if random.random() < p else 0 for _ in range(n)] for _ in range(m)]
    exp = oracle(mat)
    for s in sols:
        got = s.rowAndMaximumOnes([row[:] for row in mat])
        assert got == exp, (type(s).__name__, mat, got, exp)
print("500 ma trận ngẫu nhiên: mọi cách đều khớp oracle")

# kích thước tối đa
mat = [[random.randint(0, 1) for _ in range(100)] for _ in range(100)]
assert all(s.rowAndMaximumOnes([row[:] for row in mat]) == oracle(mat) for s in sols)
print("100x100 OK")

500 ma trận ngẫu nhiên: mọi cách đều khớp oracle
100x100 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` thay cho `>` (chọn nhầm hàng khi hoà)

In [22]:
from typing import List

def sai_ge(mat: List[List[int]]) -> List[int]:
    best_row, best_count = 0, 0
    for i, row in enumerate(mat):
        if sum(row) >= best_count:
            best_row, best_count = i, sum(row)
    return [best_row, best_count]

mat = [[0, 1], [1, 0]]
print("sai :", sai_ge(mat))                            # [1, 1]
print("đúng:", Solution().rowAndMaximumOnes(mat))      # [0, 1]

sai : [1, 1]
đúng: [0, 1]


**Vì sao sai:** với `>=`, hàng sau có cùng số `1` sẽ **ghi đè** hàng trước, kết quả là chỉ số **lớn nhất** trong các hàng hoà. Đề yêu cầu chỉ số **nhỏ nhất**. Dùng `>` (chặt).

### Lỗi 2: Khởi tạo `best_count = 0` nhưng `best_row = -1` (hoặc `None`)

In [23]:
def sai_khoi_tao(mat: List[List[int]]) -> List[int]:
    best_row, best_count = -1, 0
    for i, row in enumerate(mat):
        if sum(row) > best_count:
            best_row, best_count = i, sum(row)
    return [best_row, best_count]

print("sai :", sai_khoi_tao([[0, 0], [0, 0]]))                 # [-1, 0]
print("đúng:", Solution().rowAndMaximumOnes([[0, 0], [0, 0]]))  # [0, 0]

sai : [-1, 0]
đúng: [0, 0]


**Vì sao sai:** khi toàn ma trận là `0`, không hàng nào "hơn hẳn" `0`, nên `best_row` giữ nguyên giá trị khởi tạo `-1`. Đáp án đúng là `[0, 0]` (hàng nhỏ nhất). Khởi tạo `best_row = 0`.

### Lỗi 3: Trả về sai thứ tự `[số 1, chỉ số]`

In [24]:
def sai_thu_tu(mat: List[List[int]]) -> List[int]:
    best_row, best_count = 0, 0
    for i, row in enumerate(mat):
        if sum(row) > best_count:
            best_row, best_count = i, sum(row)
    return [best_count, best_row]       # đảo thứ tự

print("sai :", sai_thu_tu([[0, 0, 0], [0, 1, 1]]))               # [2, 1]
print("đúng:", Solution().rowAndMaximumOnes([[0, 0, 0], [0, 1, 1]]))   # [1, 2]

sai : [2, 1]
đúng: [1, 2]


**Vì sao sai:** đề yêu cầu `[chỉ số hàng, số lượng số 1]`. Với ví dụ 2, `[1, 2]` khác `[2, 1]`. Đọc kỹ thứ tự trong phần "Return".

### Lỗi 4: Dùng `max` trực tiếp trên tổng, mất chỉ số

In [25]:
def sai_mat_chi_so(mat: List[List[int]]) -> List[int]:
    m = max(sum(row) for row in mat)
    return [mat.index(m), m]          # mat.index so sánh hàng với số -> ValueError

try:
    sai_mat_chi_so([[0, 0, 0], [0, 1, 1]])
except ValueError as e:
    print("lỗi:", e)

def dung_voi_index(mat: List[List[int]]) -> List[int]:
    sums = [sum(row) for row in mat]
    m = max(sums)
    return [sums.index(m), m]         # list.index trả vị trí XUẤT HIỆN ĐẦU TIÊN

print("đúng:", dung_voi_index([[0, 0, 0], [0, 1, 1]]), dung_voi_index([[0, 1], [1, 0]]))

lỗi: 2 is not in list
đúng: [1, 2] [0, 1]


**Vì sao sai:** `mat.index(m)` tìm hàng bằng `m` (so danh sách với số) nên không bao giờ thấy. Cần tính danh sách `sums` rồi `sums.index(m)`; `index` trả về vị trí **đầu tiên**, khớp tie-break. Nhớ: `max` cho **giá trị**, `index` hoặc `enumerate` mới cho **vị trí**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hàng/cột tốt nhất theo một tiêu chí" ⇒ tính tiêu chí cho từng hàng rồi `max` (hoặc cập nhật kỷ lục khi duyệt).
- Hoà chọn chỉ số nhỏ ⇒ **so sánh chặt `>`** khi duyệt tăng dần; hoặc `max(range(m), key=...)` vì Python giữ phần tử đầu.
- Ma trận nhị phân ⇒ `sum(row)` thay cho đếm tay.
- Luôn nghĩ tới trường hợp **không ai thoả** (toàn `0`) và khởi tạo kỷ lục sao cho vẫn ra đáp án hợp lệ.
- Đọc kỹ thứ tự phần tử trong mảng trả về.

## 15. Biến thể của bài

1. **The K Weakest Rows in a Matrix (LeetCode 1337):** sắp xếp các hàng theo số `1` rồi theo chỉ số — cùng tư tưởng tie-break.
2. **Cột có nhiều số `1` nhất:** đổi hàng thành cột (`zip(*mat)`).
3. **Hàng có số `1` nhiều nhất khi mỗi hàng đã sắp xếp** (tất cả `1` dồn bên trái/phải) ⇒ có thể dùng **tìm kiếm nhị phân** để đếm số `1` mỗi hàng.
4. **Chọn hàng cuối khi hoà:** chỉ cần đổi `>` thành `>=`.
5. **Count Negative Numbers in a Sorted Matrix (LeetCode 1351).**

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp | Đếm tay từng ô | `O(m · n)` | `O(1)` | Hiểu bản chất, người mới |
| 2. `sum(row)` | Đếm bằng hàm dựng sẵn | `O(m · n)` | `O(1)` | **Nên nộp**, rõ ràng |
| 3. `max` + `key` | Tìm chỉ số có tổng lớn nhất | `O(m · n)` | `O(1)` | Ngắn gọn, dựa vào "đầu tiên thắng" |
| 4. Tuple `(sum, -i)` | Tie-break viết rõ trong dữ liệu | `O(m · n)` | `O(1)` | Khi muốn không phụ thuộc quy tắc ngầm |
| 5. Dừng sớm | Gặp hàng toàn `1` thì dừng | `O(m · n)`, thường ít hơn | `O(1)` | Khi hàng toàn `1` hay xuất hiện sớm |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[0, 1, 1].count(1)`, `sum([1, 1, 1, 0])`, `max([3, 5, 5], key=lambda x: x)`, `max(range(3), key=lambda i: [3, 5, 5][i])`.

**Bài 2 (tính tay).** Với `mat = [[1,0,1],[0,1,1],[1,1,0],[0,0,0]]`, tính số `1` từng hàng và đáp án của `rowAndMaximumOnes`.

**Bài 3 (điền TODO).** Hoàn thành hàm `count_ones_todo(row)` đếm số `1` của một hàng **không dùng** `sum` hoặc `count`.

In [26]:
def count_ones_todo(row):
    count = 0
    # TODO 1: duyệt từng phần tử x trong row
    # TODO 2: nếu x == 1 thì tăng count
    # TODO 3: return count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `row_and_max_ones_maxkey(mat)` chỉ dùng `max(range(...), key=...)` mà không nhìn Cách 3. Kiểm tra với ví dụ 1 (tie).

**Bài 5 (biến thể / mini challenge).** Viết `row_last_max_ones(mat)`: khi hoà chọn hàng có chỉ số **lớn nhất**. Thêm `col_and_max_ones(mat)`: cột có nhiều số `1` nhất (hoà chọn chỉ số cột nhỏ nhất), trả `[chỉ số cột, số 1]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `count(1)` đếm số phần tử bằng `1`; `max` trả phần tử đầu tiên khi hoà, nên chỉ số trả về là `1` (không phải `2`). Bài 2: hàng `[1,0,1]` có `2`, `[0,1,1]` có `2`, `[1,1,0]` có `2`, `[0,0,0]` có `0`.
- **Mức 2 (vừa):** Bài 3: `for x in row: if x == 1: count += 1`. Bài 4: `idx = max(range(len(mat)), key=lambda i: sum(mat[i]))`.
- **Mức 3 (sâu):** Bài 5: đổi `>` thành `>=` ở phần duyệt; cột có thể lấy bằng `zip(*mat)` rồi `enumerate`, hoặc `sum(mat[i][j] for i in range(m))` cho từng `j`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert [0, 1, 1].count(1) == 2
assert sum([1, 1, 1, 0]) == 3
assert max([3, 5, 5], key=lambda x: x) == 5
assert max(range(3), key=lambda i: [3, 5, 5][i]) == 1

# Bài 2
mat = [[1, 0, 1], [0, 1, 1], [1, 1, 0], [0, 0, 0]]
assert [sum(r) for r in mat] == [2, 2, 2, 0]
assert Solution().rowAndMaximumOnes(mat) == [0, 2]

# Bài 3
def count_ones_ans(row):
    count = 0
    for x in row:
        if x == 1:
            count += 1
    return count

assert count_ones_ans([1, 0, 1, 1]) == 3
assert count_ones_ans([0, 0]) == 0

# Bài 4
def row_and_max_ones_maxkey(mat):
    idx = max(range(len(mat)), key=lambda i: sum(mat[i]))
    return [idx, sum(mat[idx])]

assert row_and_max_ones_maxkey([[0, 1], [1, 0]]) == [0, 1]
assert row_and_max_ones_maxkey([[0, 0, 0], [0, 1, 1]]) == [1, 2]

# Bài 5
def row_last_max_ones(mat):
    best_row, best_count = 0, 0
    for i, row in enumerate(mat):
        c = sum(row)
        if c >= best_count:
            best_row, best_count = i, c
    return [best_row, best_count]

assert row_last_max_ones([[0, 1], [1, 0]]) == [1, 1]
assert row_last_max_ones([[0, 0, 0], [0, 1, 1]]) == [1, 2]

def col_and_max_ones(mat):
    best_col, best_count = 0, 0
    for j, col in enumerate(zip(*mat)):
        c = sum(col)
        if c > best_count:
            best_col, best_count = j, c
    return [best_col, best_count]

assert col_and_max_ones([[0, 1], [0, 1]]) == [1, 2]
assert col_and_max_ones([[1, 1], [1, 1]]) == [0, 2]
assert col_and_max_ones([[0, 0], [0, 0]]) == [0, 0]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (duyệt hàng, kỷ lục với `>` chặt)

```python
from typing import List

class Solution:
    def rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]:
        best_row, best_count = 0, 0
        for i, row in enumerate(mat):
            count = sum(row)
            if count > best_count:        # > chặt: hoà thì giữ hàng đầu
                best_row, best_count = i, count
        return [best_row, best_count]
```

**Checklist**

- [ ] So sánh dùng `>` (không phải `>=`) để hoà chọn chỉ số nhỏ?
- [ ] Khởi tạo `best_row = 0`, `best_count = 0` để ma trận toàn `0` ra `[0, 0]`?
- [ ] Trả về đúng thứ tự `[chỉ số hàng, số lượng số 1]`?
- [ ] Không nhầm hàng với cột (`mat[i]` là một hàng)?
- [ ] Đã thử toàn `0`, toàn `1`, hoà ở đầu, `1×1`?
- [ ] Signature đúng `rowAndMaximumOnes(self, mat: List[List[int]]) -> List[int]` và có `from typing import List`?